# Harmony Multimodel Training — Kaggle
Production path: audit → train detector → evaluate → export TorchScript/ONNX → inference API → Harmony.


In [ ]:
import os, sys, json, glob, csv, random, shutil, subprocess
from pathlib import Path
print('Python', sys.version)
print('GPU check:')
os.system('nvidia-smi --query-gpu=name,memory.total --format=csv,noheader')
print('\nKaggle inputs:')
for p in Path('/kaggle/input').glob('*'): print(' -', p)


## Required input
Attach the Harmony radiograph dataset containing the `archive/train`, `archive/valid`, and `archive/test` folders. Each split must contain `_annotations.csv` with columns `filename,width,height,class,xmin,ymin,xmax,ymax`.
Supported confirmed labels: Cavity, Fillings, Implant, Impacted Tooth.


In [ ]:
from pathlib import Path
csvs=list(Path('/kaggle/input').rglob('_annotations.csv'))
print('Annotation files found:', len(csvs))
for x in csvs: print(x)
if not csvs:
    raise FileNotFoundError('Attach the radiograph dataset first. Expected _annotations.csv under /kaggle/input.')


In [ ]:
import pandas as pd
frames=[]
for f in csvs:
    d=pd.read_csv(f); d['annotation_file']=str(f); frames.append(d)
ann=pd.concat(frames,ignore_index=True)
required={'filename','width','height','class','xmin','ymin','xmax','ymax'}
assert required.issubset(ann.columns), f'Missing columns: {required-set(ann.columns)}'
print('Rows:',len(ann),' Images:',ann.filename.nunique())
print(ann['class'].value_counts())
# Repair tiny annotation boundary errors safely: clip boxes to image bounds, then drop only degenerate boxes.
for c in ['xmin','xmax']:
    ann[c]=ann[[c,'width']].apply(lambda r: min(max(float(r[c]),0.0),float(r['width'])),axis=1)
for c in ['ymin','ymax']:
    ann[c]=ann[[c,'height']].apply(lambda r: min(max(float(r[c]),0.0),float(r['height'])),axis=1)
deg=(ann.xmax<=ann.xmin)|(ann.ymax<=ann.ymin)
print('Degenerate boxes dropped after clipping:',int(deg.sum()))
ann=ann.loc[~deg].copy()
print('Usable boxes:',len(ann))


## Training
This notebook intentionally stops before full training until the attached dataset passes the audit above. The next revision will train a torchvision detector (commercial-friendly framework path), evaluate each class, save the best checkpoint, and export ONNX for Harmony deployment.
